# 2. Pre-processing - Chuẩn hóa đầu vào

> Chạy notebook từ thư mục project hoặc để cell đầu tự dò `config.json` ở thư mục cha.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.json").exists())
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

Project root: D:\setup\BTL01_Nhom_HoTen_MSSV\BTL01_Nhom_HoTen_MSSV


## 2.1 Quyết định tiền xử lý
Không mã hóa nhãn hay loại bỏ biến. Chỉ chuẩn hóa x1, x2 bằng mean/std được fit trên train. Validation và test chỉ gọi `transform`, không được dùng để fit scaler.

In [2]:
import pandas as pd
from src.mlp_core import InputStandardizer
tr = pd.read_csv(ROOT / "data/train.csv")
va = pd.read_csv(ROOT / "data/validation.csv")
te = pd.read_csv(ROOT / "data/test.csv")
xtr = tr[["x1", "x2"]].to_numpy(dtype="float32")
scaler = InputStandardizer().fit(xtr)
print("train-only scaler:", scaler.state_dict())
for name, df in [("train", tr), ("validation", va), ("test", te)]:
    x = df[["x1", "x2"]].to_numpy(dtype="float32")
    z = scaler.transform(x)
    print(name, "standardized mean:", z.mean(axis=0).round(4), "std:", z.std(axis=0).round(4))

train-only scaler: {'mean': [0.030335277318954468, -0.028754904866218567], 'std': [1.1517897844314575, 2.2772159576416016]}
train standardized mean: [-0.  0.] std: [1. 1.]
validation standardized mean: [-0.0451  0.0646] std: [1.0155 0.9969]
test standardized mean: [-0.2046 -0.0117] std: [1.0286 1.0197]


## 2.2 Xuất dữ liệu chuẩn hóa
Các biến chuẩn hóa được lưu trong `exps_/feature1/`. Không chuẩn hóa y để giữ thang đo mật độ tự nhiên.

In [3]:
import json
from pathlib import Path
out = ROOT / "exps_/feature1"; out.mkdir(parents=True, exist_ok=True)
(out / "input_standardizer.json").write_text(json.dumps(scaler.state_dict(), indent=2), encoding="utf-8")
for name, df in [("train", tr), ("validation", va), ("test", te)]:
    x = scaler.transform(df[["x1", "x2"]].to_numpy(dtype="float32"))
    pd.DataFrame({"x1_scaled":x[:,0], "x2_scaled":x[:,1], "y":df.y}).to_csv(out / f"{name}_standardized.csv", index=False)
print("Saved to", out)

Saved to D:\setup\BTL01_Nhom_HoTen_MSSV\BTL01_Nhom_HoTen_MSSV\exps_\feature1
